<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_38.84873.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 81% 75.0M/92.7M [00:00<00:00, 194MB/s]
100% 92.7M/92.7M [00:00<00:00, 190MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [ ]:
!pip install dask-cuda==24.12.0

In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

In [ ]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [ ]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [ ]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [ ]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)

In [ ]:
train = pd.concat([train,train_ex], axis = 0, ignore_index = True)

In [ ]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
oof = np.zeros(len(train))
preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(
                              learning_rate = 0.11509572776170199,
                              l2_leaf_reg=5,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)

    oof[test_idx] = val_preds

    preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE', 'colorxweight', 'Sizexweight',
       'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870457	total: 88ms	remaining: 1m 27s
250:	learn: 38.6241243	total: 11.5s	remaining: 34.4s
500:	learn: 38.5952857	total: 21.7s	remaining: 21.6s
750:	learn: 38.5708344	total: 31.8s	remaining: 10.6s
999:	learn: 38.5475938	total: 41.9s	remaining: 0us
Fold: 1, Score: 38.63922340533008
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Sty

In [ ]:
plt.figure(figsize=(10, 10))
sns.barplot(x=model.get_feature_importance(), y=x_train.columns)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

In [ ]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = preds
submission.to_csv("submission.csv", index = False)

submission

In [ ]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission with te median and var no hyperparameter tuning'

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
!kaggle competitions submissions -c {competition}